In [1]:
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from Bio import SeqIO
from tqdm import tqdm
import os

df1 = pd.read_csv('/home/wuke/project/bio_deeplearning/Z-Benchmark-不同模型在独立数据集合的性能比较/new_reordered_df1.csv')
df_p = pd.read_csv('../../Results/predictions.csv')
fasta_file_path = '/home/wuke/project/bio_deeplearning/zzz_benchmark/data/bingxue_seq.fasta'

sequence_to_index = {}
for record in SeqIO.parse(fasta_file_path, "fasta"):
    seq_id = record.id.split('_')[-1]
    sequence = str(record.seq)
    sequence_to_index[sequence] = seq_id
df1['Index'] = None
for index, row in tqdm(df1.iterrows(), total=df1.shape[0]):
    smiles = row['Smiles']
    sequence = row['Sequence']
    Kcat = row['Value']
    if sequence in sequence_to_index:
        df1.at[index, 'Index'] = sequence_to_index[sequence]
        pdb_filename = f'seq_{sequence_to_index[sequence]}.pdb'
        structure_path = os.path.join('/home/wuke/project/bio_deeplearning/zzz_benchmark/data/bingxue_pdb', pdb_filename)
        if not os.path.exists(structure_path):
            print(f"Warning: No matching structure found for entry at index {sequence_to_index[sequence]}")
            structure_path = None
    else:
        print(f"Error: No matching sequence found for entry at seq {index+1}")
        print(sequence)
        structure_path = None
    if "." not in smiles and float(Kcat) > 0 and structure_path is not None:
        pass
    else:
        df1.drop(index, inplace=True)
df1_filtered = df1

assert len(df1_filtered) == len(df_p), "数据量不一致"

df_p['exp2'] = np.exp2(df_p['Prediction'].values)

df1_filtered['Prediction'] = df_p['exp2'].values
df1_filtered.to_csv('DeepEnzyme_prediction.csv', index=False)

 26%|██▌       | 8873/34140 [00:01<00:04, 5580.54it/s]

 38%|███▊      | 13069/34140 [00:02<00:02, 9054.40it/s]

 46%|████▌     | 15691/34140 [00:02<00:02, 7615.21it/s]

 60%|██████    | 20524/34140 [00:03<00:02, 5607.33it/s]

 84%|████████▎ | 28510/34140 [00:05<00:00, 5705.72it/s]

 96%|█████████▌| 32797/34140 [00:06<00:00, 5438.74it/s]

100%|██████████| 34140/34140 [00:06<00:00, 5488.21it/s]
